# MoMA Collection Analysis
This notebook explores the Museum of Modern Art (MoMA) artwork dataset.

We analyze artworks by creation and acquisition dates, and classify trends over time using various visual encodings like bar plots, line plots, heatmaps, and stacked area charts.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from utils.data_frames import artworks
from utils.utils import (
    clean_years,
    clean_classification_dataframe,
    get_most_in_dataframe,
    filter_by_amount
)

from artworks import (
    ArtworkCols,
    entry_time_series_barplot,
    overlay_time_series_lineplot,
)

from classification import (
    classification_by_year_lineplot,
    all_classification_by_year_heatmap,
    all_classification_by_year_stack,
    all_classification_by_year_lineplot
)

## Overview of Entries by Date

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 6))

# Extract and clean date fields
entries_created_by_year = clean_years(artworks[ArtworkCols.Date.value]).dropna().value_counts().sort_index()
entries_acquired_by_year = clean_years(artworks[ArtworkCols.DateAcquired.value]).dropna().value_counts().sort_index()

# Group by decade
entries_created_by_decade = entries_created_by_year.groupby((entries_created_by_year.index // 10) * 10).sum()

# Plot
entry_time_series_barplot(entries_created_by_decade, title="Entries Created by Decade", ax=axs[0, 0])
overlay_time_series_lineplot(entries_created_by_year, entries_acquired_by_year, title="Overlayed Created and Acquired by Year", ax=axs[0, 1])

## Classification Time Series by Creation vs Acquisition Date

In [ ]:
# Filter out archive entries
valid_classifications = artworks[ArtworkCols.Classification.value].apply(
    lambda x: x not in ["Mies van der Rohe Archive", "Frank Lloyd Wright Archive"]
)
filtered_artworks = artworks[valid_classifications]

# Extract classification & date columns
classifications_by_date = filtered_artworks[[ArtworkCols.Classification.value, ArtworkCols.Date.value]]
classifications_by_date_acquired = filtered_artworks[[ArtworkCols.Classification.value, ArtworkCols.DateAcquired.value]]

# Clean and convert to matrices
classifications_by_date_matrix = clean_classification_dataframe(classifications_by_date, ArtworkCols.Date.value)
classifications_by_date_acquired_matrix = clean_classification_dataframe(classifications_by_date_acquired, ArtworkCols.DateAcquired.value)

## Most Represented Classifications Over Time

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 6))

# Top 5 classifications
head = 5
top5_classifications_by_date_created = get_most_in_dataframe(classifications_by_date_matrix, head)
top5_classifications_by_date_acquired = get_most_in_dataframe(classifications_by_date_acquired_matrix, head)

classification_by_year_lineplot(classifications_by_date_matrix, top5_classifications_by_date_created, title="Most represented classifications by year", ax=axs[1, 0])
classification_by_year_lineplot(classifications_by_date_acquired_matrix, top5_classifications_by_date_acquired, title="Most represented classifications by date acquired", ax=axs[1, 1])

plt.tight_layout()
plt.show()

## Distribution of Classifications Over Time

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(14, 8))

filtered_classifications_by_date_acquired_matrix = filter_by_amount(classifications_by_date_acquired_matrix, 1000)
filtered_entries_classifications_by_date_matrix = filter_by_amount(classifications_by_date_matrix, 1000)

all_classification_by_year_lineplot(
    filtered_entries_classifications_by_date_matrix,
    filtered_entries_classifications_by_date_matrix.columns,
    title="Yearly Distribution of Artwork Classifications by Creation Date",
    ax=axs[0]
)

plt.tight_layout()
plt.show()

## Classification Heatmap (Creation Date)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

all_classification_by_year_heatmap(
    classifications_by_date_matrix,
    "Classification heatmap, proportional to occurrence count",
    ax=ax
)

plt.tight_layout()
plt.show()

## Classification Stacked Area Charts (Acquisition Date, Smoothed)

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(14, 10))
rolling_window = 2
smoothed = filtered_classifications_by_date_acquired_matrix.rolling(window=rolling_window).mean()

all_classification_by_year_stack(smoothed, "minmax", "Classifications by Date Acquired Stacked Area Chart (MinMax)", ax=axs[0])
all_classification_by_year_stack(smoothed, "mean", "Classifications by Date Acquired Stacked Area Chart (Z-score)", ax=axs[1])
all_classification_by_year_stack(smoothed, "proportional", "Classifications by Date Acquired Stacked Area Chart (Proportional)", ax=axs[2])

plt.tight_layout()
plt.show()

## Case Study: Post-modernist Classifications

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 6))

post_modernist_classifications = ["Media", "Audio", "Video", "Multiple", "Installation", "Digital", "Ephemera", "Performance"]

classification_by_year_lineplot(
    classifications_by_date_matrix,
    post_modernist_classifications,
    title="Post-modernist classifications by Year",
    ax=axs[1, 0],
)

classification_by_year_lineplot(
    classifications_by_date_acquired_matrix,
    post_modernist_classifications,
    title="Post-modernist Classifications by Date Acquired",
    ax=axs[1, 1],
)

plt.tight_layout()
plt.show()